# PATSTAT Disruption + F/E/G — CD / F,E,G / ni,nj,nk per window

The twin of `PatentView/notebook/patent_disruption.ipynb`, on the application-level network from
`patstat_reference.parquet`, with the **numba engine of the OpenAlex / Dimensions notebooks** (the
PatentView per-focal Python loop takes hours on 8.5 M patents; the universe here is an order of magnitude
larger).

## Graph
Directed edges (citing application -> cited application) from `patstat_reference` under `ps.EDGE_WHERE`
(`age >= 0`, not replenished), **de-duplicated** (many publications carry the same reference) with
self-loops already removed upstream. Node year = filing year. Cached as `cache/patstat_csr.npz`.

## Metrics
For focal $P$, window $W$: $A_W$ = citers within $W$ years, $\mathrm{refs}(P)$, and for each citer $C$:
$up_C = |\mathrm{refs}(P) \cap \mathrm{refs}(C)|$, $down_C = |A_W \cap \mathrm{refs}(C)|$.
$n_j = \#\{C \in A_W: up_C > 0\}$, $n_i = |A_W| - n_j$, $n_k = |B_W| - n_j$ with $B_W$ = applications
citing a reference of $P$ within $W$; $CD_W = (n_i - n_j)/(n_i + n_j + n_k)$.
F / E / G per citer: $up = down = 0 \Rightarrow$ G; $up > down \Rightarrow$ E; $down > up \Rightarrow$ F;
ties split 0.5 / 0.5. $F + E + G = 1$.

## Output
`PATSTAT/output/patstat_disruption.parquet` — `appln_id` + `CD/F/E/G/ni/nj/nk` x `{_3,_5,_10,_all}`.
Applications with no citer: NaN (CD/F/E/G) / -1 (ni/nj/nk). Only applications that appear in the edge list
(as citer or cited) are rows.

In [ ]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
OUT_FP = ps.out('patstat_disruption.parquet')
ps.preflight('patstat_disruption')

REF, META = ps.out('patstat_reference.parquet'), ps.out('patstat_metadata.parquet')
CSR_NPZ = ps.cache('patstat_csr.npz')
WINS = np.array([3, 5, 10, 2_000_000_000], dtype=np.int64)
SFX  = ['_3', '_5', '_10', '_all']

## 1. Graph -> CSR (cached)

In [ ]:
%%time
# 1. Graph -> integer codes -> CSR (cached)
if os.path.exists(CSR_NPZ):
    z = np.load(CSR_NPZ)
    out_ptr, out_idx, in_ptr, in_idx, year, uni = z['out_ptr'], z['out_idx'], z['in_ptr'], z['in_idx'], z['year'], z['uni']
    print(f'CSR cache: {CSR_NPZ}')
else:
    con = ps.connect()
    e = con.execute(f"""SELECT DISTINCT citing_id, cited_id FROM read_parquet('{REF}') WHERE {ps.EDGE_WHERE}""").fetchnumpy()
    c_from_id, c_to_id = e['citing_id'].astype(np.int64), e['cited_id'].astype(np.int64); del e
    print(f'{len(c_from_id):,} distinct directed edges')
    uni = np.unique(np.concatenate([c_from_id, c_to_id]))
    n = len(uni)
    c_from = np.searchsorted(uni, c_from_id).astype(np.int64); c_to = np.searchsorted(uni, c_to_id).astype(np.int64)
    del c_from_id, c_to_id; gc.collect()
    yr = con.execute(f"SELECT appln_id, filing_year FROM read_parquet('{META}')").fetchnumpy(); con.close()
    pos = np.searchsorted(yr['appln_id'], uni) if np.all(np.diff(yr['appln_id']) > 0) else None
    if pos is None:
        order = np.argsort(yr['appln_id']); ids = yr['appln_id'][order]; fys = yr['filing_year'][order]
    else:
        ids, fys = yr['appln_id'], yr['filing_year']
    pos = np.searchsorted(ids, uni); assert np.array_equal(ids[pos], uni), 'every node must be in the metadata universe'
    year = fys[pos].astype(np.int32); del yr, ids, fys; gc.collect()
    def csr(src, dst):
        order = np.argsort(src, kind='stable'); idx = dst[order].astype(np.int32); del order
        ptr = np.zeros(n + 1, np.int64); np.add.at(ptr, src + 1, 1); np.cumsum(ptr, out=ptr); return ptr, idx
    out_ptr, out_idx = csr(c_from, c_to); in_ptr, in_idx = csr(c_to, c_from)
    del c_from, c_to; gc.collect()
    np.savez(CSR_NPZ, out_ptr=out_ptr, out_idx=out_idx, in_ptr=in_ptr, in_idx=in_idx, year=year, uni=uni)
    print(f'CSR built and cached -> {CSR_NPZ}')
n = len(year)
print(f'CSR: {n:,} applications, {len(out_idx):,} edges; filing years {year.min()}-{year.max()}')

## 2. Numba engine (verbatim from the OpenAlex / Dimensions notebooks)

In [ ]:
from numba import njit, prange

@njit(inline='always')
def _bfind(arr, x):
    lo = 0; hi = len(arr)
    while lo < hi:
        mid = (lo + hi) >> 1
        if arr[mid] < x: lo = mid + 1
        else: hi = mid
    return lo < len(arr) and arr[lo] == x

@njit(parallel=True)
def compute_numba(focal, out_ptr, out_idx, in_ptr, in_idx, year, wins,
                  CD, Ff, Ef, Gf, NI, NJ, NK):
    W = len(wins)
    for t in prange(len(focal)):
        F = focal[t]
        a0 = in_ptr[F]; a1 = in_ptr[F + 1]
        if a1 == a0:
            continue
        yF = year[F]
        A = in_idx[a0:a1]
        R = out_idx[out_ptr[F]:out_ptr[F + 1]]
        Rs = np.sort(R); As = np.sort(A)
        Nw = np.zeros(W, np.int64); njw = np.zeros(W, np.int64)
        cext = np.zeros(W, np.int64); cfnd = np.zeros(W, np.int64)
        tiew = np.zeros(W, np.int64); cgw = np.zeros(W, np.int64); Bw = np.zeros(W, np.int64)
        for ci in range(len(A)):
            c = A[ci]; dc = year[c] - yF
            if dc < 0:
                continue
            up = 0; downc = np.zeros(W, np.int64)
            for ri in range(out_ptr[c], out_ptr[c + 1]):
                d = out_idx[ri]
                if _bfind(Rs, d): up += 1
                if _bfind(As, d):
                    dd = year[d] - yF
                    if dd >= 0:
                        for k in range(W):
                            if dd <= wins[k]: downc[k] += 1
            for k in range(W):
                if dc <= wins[k]:
                    Nw[k] += 1; dk = downc[k]
                    if up > 0: njw[k] += 1
                    if up > dk: cext[k] += 1
                    elif dk > up: cfnd[k] += 1
                    elif up == dk and up > 0: tiew[k] += 1
                    elif up == 0 and dk == 0: cgw[k] += 1
        totB = 0
        for ri in range(len(R)):
            r = R[ri]; totB += in_ptr[r + 1] - in_ptr[r]
        if totB > 0:
            buf = np.empty(totB, np.int32); p = 0
            for ri in range(len(R)):
                r = R[ri]
                for j in range(in_ptr[r], in_ptr[r + 1]):
                    buf[p] = in_idx[j]; p += 1
            buf.sort(); prev = np.int32(-1)
            for ii in range(totB):
                b = buf[ii]
                if b == prev or b == F: continue
                prev = b; dd = year[b] - yF
                if dd >= 0:
                    for k in range(W):
                        if dd <= wins[k]: Bw[k] += 1
        for k in range(W):
            Nk = Nw[k]; Bk = Bw[k]
            if Nk == 0 and Bk == 0: continue
            njk = njw[k]; nik = Nk - njk; nkk = Bk - njk; denom = nik + njk + nkk
            NI[k, F] = nik; NJ[k, F] = njk; NK[k, F] = nkk
            CD[k, F] = (nik - njk) / denom if denom > 0 else np.nan
            if Nk > 0:
                Ef[k, F] = (cext[k] + 0.5 * tiew[k]) / Nk
                Ff[k, F] = (cfnd[k] + 0.5 * tiew[k]) / Nk
                Gf[k, F] = cgw[k] / Nk
print('engine ready')

## 3. Run over all focal applications with citers

In [ ]:
%%time
W = len(WINS)
CD = np.full((W, n), np.nan, np.float32); Ff = np.full((W, n), np.nan, np.float32)
Ef = np.full((W, n), np.nan, np.float32); Gf = np.full((W, n), np.nan, np.float32)
NI = np.full((W, n), -1, np.int32); NJ = np.full((W, n), -1, np.int32); NK = np.full((W, n), -1, np.int32)
focal_all = np.flatnonzero(in_ptr[1:] - in_ptr[:-1] > 0).astype(np.int64)
print(f'focal with citers: {len(focal_all):,}  -- warm-up compile...')
compute_numba(focal_all[:5000], out_ptr, out_idx, in_ptr, in_idx, year, WINS, CD, Ff, Ef, Gf, NI, NJ, NK)
CD[:] = np.nan; Ff[:] = np.nan; Ef[:] = np.nan; Gf[:] = np.nan; NI[:] = -1; NJ[:] = -1; NK[:] = -1
tc = time.time(); print('running full numba compute...')
compute_numba(focal_all, out_ptr, out_idx, in_ptr, in_idx, year, WINS, CD, Ff, Ef, Gf, NI, NJ, NK)
print(f'compute done in {time.time()-tc:.0f}s')

## 4. Save

In [ ]:
cols = {'appln_id': uni.astype(np.int64)}
for k, s in enumerate(SFX):
    cols[f'CD{s}'] = CD[k]; cols[f'F{s}'] = Ff[k]; cols[f'E{s}'] = Ef[k]; cols[f'G{s}'] = Gf[k]
    cols[f'ni{s}'] = NI[k]; cols[f'nj{s}'] = NJ[k]; cols[f'nk{s}'] = NK[k]
out = pd.DataFrame(cols)
out.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(out):,} rows, {len(out.columns)} cols)')
for k, s in enumerate(SFX):
    print(f'  CD{s}: defined {np.isfinite(CD[k]).mean()*100:5.1f}%  mean {np.nanmean(CD[k]):+.4f}  |  '
          f'f/e/g = {np.nanmean(Ff[k]):.3f}/{np.nanmean(Ef[k]):.3f}/{np.nanmean(Gf[k]):.3f}')
ok = np.isfinite(Ff[1]); assert np.allclose(Ff[1][ok] + Ef[1][ok] + Gf[1][ok], 1, atol=1e-4), 'F+E+G must be 1'
display(out[np.isfinite(out.CD_5)].head(8))